In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


In [2]:
import os
csv_path = 'demand_forecasting.csv'
if not os.path.exists(csv_path):
    csv_path = '/kaggle/input/datasets/raminhuseyn/demand-forecasting-dataset/demand_forecasting.csv'
df = pd.read_csv(csv_path)
df.head(3)


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,Weather Condition,Promotion,Competitor Pricing,Seasonality,Epidemic,Demand
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,Snowy,0,85.73,Winter,0,115
1,2022-01-01,S001,P0002,Clothing,North,117,117,249,80.16,15,Snowy,1,92.02,Winter,0,229
2,2022-01-01,S001,P0003,Clothing,North,247,114,612,62.94,10,Snowy,1,60.08,Winter,0,157


In [3]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76000 entries, 0 to 75999
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                76000 non-null  object 
 1   Store ID            76000 non-null  object 
 2   Product ID          76000 non-null  object 
 3   Category            76000 non-null  object 
 4   Region              76000 non-null  object 
 5   Inventory Level     76000 non-null  int64  
 6   Units Sold          76000 non-null  int64  
 7   Units Ordered       76000 non-null  int64  
 8   Price               76000 non-null  float64
 9   Discount            76000 non-null  int64  
 10  Weather Condition   76000 non-null  object 
 11  Promotion           76000 non-null  int64  
 12  Competitor Pricing  76000 non-null  float64
 13  Seasonality         76000 non-null  object 
 14  Epidemic            76000 non-null  int64  
 15  Demand              76000 non-null  int64  
dtypes: f

In [4]:
df.isnull().sum().sum()


np.int64(0)

In [5]:
df.duplicated().sum()


np.int64(0)

In [6]:
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')

In [9]:
pip install holidays

     ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
      --------------------------------------- 0.0/1.5 MB ? eta -:--:--
     -- ------------------------------------- 0.1/1.5 MB 1.1 MB/s eta 0:00:02
     --- ------------------------------------ 0.1/1.5 MB 1.1 MB/s eta 0:00:02
     ----- ---------------------------------- 0.2/1.5 MB 1.1 MB/s eta 0:00:02
     ------------ --------------------------- 0.5/1.5 MB 2.0 MB/s eta 0:00:01
     ------------- -------------------------- 0.5/1.5 MB 2.1 MB/s eta 0:00:01
     ----------------------- ---------------- 0.9/1.5 MB 2.8 MB/s eta 0:00:01
     ---------------------------------------  1.5/1.5 MB 4.0 MB/s eta 0:00:01
     ---------------------------------------- 1.5/1.5 MB 3.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import holidays

df['Date'] = pd.to_datetime(df['Date'])

# Sort by series and date to compute lags and rolling features correctly
df = df.sort_values(by=['Store ID', 'Product ID', 'Date']).reset_index(drop=True)

# Lags of Target
df['lag_1'] = df.groupby(['Store ID', 'Product ID'])['Demand'].shift(1)
df['lag_7'] = df.groupby(['Store ID', 'Product ID'])['Demand'].shift(7)
df['lag_30'] = df.groupby(['Store ID', 'Product ID'])['Demand'].shift(30)

# Rolling window statistics (shift by 1 day first to prevent data leakage)
df['rolling_mean_7'] = df.groupby(['Store ID', 'Product ID'])['Demand'].transform(lambda x: x.shift(1).rolling(7).mean())
df['rolling_std_7'] = df.groupby(['Store ID', 'Product ID'])['Demand'].transform(lambda x: x.shift(1).rolling(7).std())

df['rolling_mean_14'] = df.groupby(['Store ID', 'Product ID'])['Demand'].transform(lambda x: x.shift(1).rolling(14).mean())
df['rolling_std_14'] = df.groupby(['Store ID', 'Product ID'])['Demand'].transform(lambda x: x.shift(1).rolling(14).std())

df['rolling_mean_30'] = df.groupby(['Store ID', 'Product ID'])['Demand'].transform(lambda x: x.shift(1).rolling(30).mean())
df['rolling_std_30'] = df.groupby(['Store ID', 'Product ID'])['Demand'].transform(lambda x: x.shift(1).rolling(30).std())

# Day of week and weekend features
df['day_of_week'] = df['Date'].dt.dayofweek
df['is_weekend'] = df['Date'].dt.dayofweek.isin([5, 6]).astype(int)

# Indian holidays feature
in_holidays = holidays.country_holidays('IN')
df['is_holiday'] = df['Date'].apply(lambda x: 1 if x in in_holidays else 0)

# Trend feature (days since start date)
start_date = df['Date'].min()
df['trend'] = (df['Date'] - start_date).dt.days

# Pricing & supply features from original pipeline
df['Total_Earnings'] = df['Units Sold'] * df['Price'] * (1 - df['Discount']/100)
df['Products_to_sell'] = df['Units Ordered'] - df['Units Sold']

df['Price_gap_pct'] = (df['Price'] - df['Competitor Pricing']) / df['Competitor Pricing']
df['Discount_effect'] = df['Discount'] * df['Promotion']

# Inventory features — use inventory-to-demand ratio and days-of-supply instead of
# Inventory / (Inventory + 1), which saturates near 1 for any level above ~50.
df['inventory_demand_ratio'] = df['Inventory Level'] / (df['rolling_mean_7'] + 1e-6)
df['days_of_supply'] = df['Inventory Level'] / (df['rolling_mean_30'] + 1e-6)

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day

# Drop rows with NaNs resulting from shifts and rolling windows (first 30 days)
df = df.dropna().reset_index(drop=True)

# Sort primarily by Date and secondarily by Store ID and Product ID for chronological validation
df = df.sort_values(by=['Date', 'Store ID', 'Product ID']).reset_index(drop=True)


In [11]:
numeric_cols = df.select_dtypes(include="number").columns
object_cols = df.select_dtypes(include="object").columns

print("🔢 NUMERIC COLUMNS")
print("-" * 50)

for col in numeric_cols:
    nunique = df[col].nunique()
    print(f"\n📌 {col}")
    print(f"nunique: {nunique}")
    
    if nunique < 15:
        print("unique values:", df[col].unique())

print("\n" + "=" * 60)

print("🔤 OBJECT COLUMNS")
print("-" * 50)

for col in object_cols:
    nunique = df[col].nunique()
    print(f"\n📌 {col}")
    print(f"nunique: {nunique}")
    
    if nunique < 15:
        print("unique values:", df[col].unique())


🔢 NUMERIC COLUMNS
--------------------------------------------------

📌 Inventory Level
nunique: 1414

📌 Units Sold
nunique: 329

📌 Units Ordered
nunique: 993

📌 Price
nunique: 15288

📌 Discount
nunique: 6
unique values: [10  5  0 15 20 25]

📌 Promotion
nunique: 2
unique values: [0 1]

📌 Competitor Pricing
nunique: 15836

📌 Epidemic
nunique: 2
unique values: [0 1]

📌 Demand
nunique: 339

📌 lag_1
nunique: 339

📌 lag_7
nunique: 339

📌 lag_30
nunique: 340

📌 rolling_mean_7
nunique: 1346

📌 rolling_std_7
nunique: 71339

📌 rolling_mean_14
nunique: 2290

📌 rolling_std_14
nunique: 72161

📌 rolling_mean_30
nunique: 4152

📌 rolling_std_30
nunique: 72358

📌 day_of_week
nunique: 7
unique values: [0 1 2 3 4 5 6]

📌 is_weekend
nunique: 2
unique values: [0 1]

📌 is_holiday
nunique: 2
unique values: [0 1]

📌 trend
nunique: 730

📌 Total_Earnings
nunique: 70480

📌 Products_to_sell
nunique: 1137

📌 Price_gap_pct
nunique: 72482

📌 Discount_effect
nunique: 5
unique values: [ 0 15 10 20 25]

📌 inventory_de

<# d# i# v#  # s# t# y# l# e# =# "# 
b# a# c# k# g# r# o# u# n# d# :#  # l# i# n# e# a# r# -# g# r# a# d# i# e# n# t# (# 1# 3# 5# d# e# g# ,#  # ## 0# 2# 0# 6# 1# 7#  # 0# %# ,#  # ## 0# 2# 0# 6# 1# 7#  # 1# 0# 0# %# )# ;# 
b# o# r# d# e# r# -# r# a# d# i# u# s# :#  # 1# 4# p# x# ;# 
p# a# d# d# i# n# g# :#  # 1# 8# p# x#  # 2# 4# p# x# ;# 
m# a# r# g# i# n# :#  # 2# 8# p# x#  # 0# ;# 
b# o# x# -# s# h# a# d# o# w# :#  # 0#  # 1# 2# p# x#  # 3# 0# p# x#  # r# g# b# a# (# 0# ,# 0# ,# 0# ,# 0# .# 6# )# ;# 
f# o# n# t# -# f# a# m# i# l# y# :#  # A# r# i# a# l# ,#  # H# e# l# v# e# t# i# c# a# ,#  # s# a# n# s# -# s# e# r# i# f# ;# 
"# ># 

<# h# 2#  # s# t# y# l# e# =# "# 
m# a# r# g# i# n# :#  # 0# ;# 
f# o# n# t# -# s# i# z# e# :#  # 2# 6# p# x# ;# 
f# o# n# t# -# w# e# i# g# h# t# :#  # 7# 0# 0# ;# 
c# o# l# o# r# :#  # ## e# 5# e# 7# e# b# ;# 
l# e# t# t# e# r# -# s# p# a# c# i# n# g# :#  # 0# .# 6# p# x# ;# 
"# ># 
📊#  # E# D# A#  # &#  # V# i# s# u# a# l# i# z# a# t# i# o# n# 
<# /# h# 2# ># 

<# /# d# i# v# >


In [12]:
plt.figure(figsize=(14,6))

sns.lineplot(data=df, x='Date', y='Units Sold', label='Units Sold', color='red') 
sns.lineplot(data=df, x='Date', y='Units Ordered', label='Units Ordered', color='#0D47A1')  

plt.title("Units Sold vs Units Ordered Over Time", fontsize=14)
plt.xlabel("Date")
plt.ylabel("Units")
plt.grid(alpha=0.2)

plt.legend()
plt.tight_layout()
plt.show()


In [13]:
plt.figure(figsize=(10,6))

sns.scatterplot(
    data=df,
    x='Price_gap_pct',
    y='Demand',
    hue='Promotion',
    alpha=0.6
)

plt.title("Demand vs Price Gap (Competitive Effect)")
plt.grid(alpha=0.2)
plt.show()


In [14]:
cols_for_count = ['Category','Region','Weather Condition','Seasonality','Discount','Promotion','Epidemic']
sns.set_style("darkgrid")

fig, axes = plt.subplots(4, 2, figsize=(14, 12))
axes = axes.flatten()

for i, col in enumerate(cols_for_count):
    ax = sns.countplot(
        data=df,
        x=col,
        ax=axes[i],
        palette="Set2")
    
    axes[i].set_title(f"{col} Distribution", fontsize=12, weight="bold")
    axes[i].tick_params(axis='x', rotation=30)

    for p in ax.patches:
        height = p.get_height()
        ax.annotate(
            f"{int(height)}",
            (p.get_x() + p.get_width() / 2., height),
            ha="center",
            va="bottom",
            fontsize=9)

if len(cols_for_count) < len(axes):
    for j in range(len(cols_for_count), len(axes)):
        fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


In [15]:
categorical_cols = ['Category', 'Region', 'Weather Condition', 'Seasonality']

plt.style.use("dark_background")

fig, axes = plt.subplots(2, 2, figsize=(14,10))
axes = axes.flatten()  

for i, col in enumerate(categorical_cols):
    
    n_colors = df[col].nunique()
    palette = sns.color_palette("husl", n_colors)
    
    sns.scatterplot(
        data=df,
        x='Units Sold',
        y='Demand',
        hue=col,
        palette=palette,
        alpha=0.7,
        ax=axes[i]
    )
    
    axes[i].set_title(f"{col}")
    axes[i].grid(alpha=0.2)
    
    axes[i].legend(title=col, fontsize=8)

plt.tight_layout()
plt.show()


In [16]:
plt.style.use("dark_background")

fig, axes = plt.subplots(1, 2, figsize=(16,6))

ct_cat = pd.crosstab(df['Category'], df['Discount'])
ct_cat = ct_cat.sort_index(axis=1)
ct_cat_pct = ct_cat.div(ct_cat.sum(axis=1), axis=0) * 100

ax1 = ct_cat_pct.plot(
    kind='bar',
    stacked=True,
    colormap='viridis',
    ax=axes[0]
)

for container in ax1.containers:
    ax1.bar_label(container, fmt='%.1f%%', label_type='center', fontsize=8)

ax1.set_title("Discount Distribution by Category (%)")
ax1.set_xlabel("Category")
ax1.set_ylabel("Percentage")
ax1.legend(title="Discount", fontsize=8)


ct_season = pd.crosstab(df['Seasonality'], df['Discount'])
ct_season = ct_season.sort_index(axis=1)

order = ['Winter', 'Spring', 'Summer', 'Autumn']
ct_season = ct_season.loc[order]

ct_season_pct = ct_season.div(ct_season.sum(axis=1), axis=0) * 100

ax2 = ct_season_pct.plot(
    kind='bar',
    stacked=True,
    colormap='viridis',
    ax=axes[1]
)

for container in ax2.containers:
    ax2.bar_label(container, fmt='%.1f%%', label_type='center', fontsize=8)

ax2.set_title("Discount Distribution by Season (%)")
ax2.set_xlabel("Season")
ax2.set_ylabel("Percentage")
ax2.legend(title="Discount", fontsize=8)


plt.tight_layout()
plt.show()


In [17]:
plt.figure(figsize=(8,5))

season_avg = df.groupby('Seasonality')['Total_Earnings'].mean().sort_values()

ax = sns.barplot(
    x=season_avg.index,
    y=season_avg.values,
    palette="viridis"
)

for container in ax.containers:
    ax.bar_label(container, fmt='%.0f', label_type='center')

plt.title("Average Total Earnings by Season")
plt.ylabel("Avg Earnings")

plt.tight_layout()
plt.show()


In [18]:
plt.figure(figsize=(14,6))

sns.lineplot(
    data=df,
    x='Date',
    y='Total_Earnings',
    hue='Seasonality',
    palette='husl'
)

plt.title("Total Earnings Over Time by Season")
plt.grid(alpha=0.2)

plt.tight_layout()
plt.show()


In [19]:
plt.figure(figsize=(8,5))

sns.histplot(df['Products_to_sell'], bins=50, kde=True, color='orange')

plt.title("Products_to_sell Distribution")
plt.grid(alpha=0.2)

plt.show()


In [20]:
plt.figure(figsize=(8,5))

sns.scatterplot(
    data=df,
    x='Demand',
    y='Products_to_sell',
    hue='Seasonality',
    alpha=0.6
)

plt.title("Demand vs Products to Sell (Stock Gap Analysis)")
plt.xlabel("Demand")
plt.ylabel("Products to Sell")

plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()


In [21]:
plt.style.use("default")

corr = df.select_dtypes(include='number').corr()

mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(10,8))

sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="viridis",
    center=0,
    linewidths=0.5
)

plt.title("Correlation Heatmap (Lower Triangle)")
plt.tight_layout()
plt.show()


In [22]:
df_model = df.copy()

leakage_cols = [
    'Units Sold',
    'Units Ordered',
    'Products_to_sell',
    'Total_Earnings',
    'Price'
 ]

df_model = df_model.drop(columns=[c for c in leakage_cols if c in df_model.columns])

# Reduce redundancy among highly collinear price features while keeping a stable signal feature.
price_like_cols = ['Price_diff', 'Price_ratio', 'Price_gap_pct']
available_price_cols = [c for c in price_like_cols if c in df_model.columns]

if len(available_price_cols) > 1:
    keep_col = 'Price_gap_pct' if 'Price_gap_pct' in available_price_cols else available_price_cols[0]
    drop_cols = [c for c in available_price_cols if c != keep_col]
    df_model = df_model.drop(columns=drop_cols)
    print(f"Dropped redundant price features: {drop_cols}; kept {keep_col}")

df_model = df_model.sort_values(['Date', 'Store ID', 'Product ID']).reset_index(drop=True)

<# d# i# v#  # s# t# y# l# e# =# "# 
b# a# c# k# g# r# o# u# n# d# :#  # l# i# n# e# a# r# -# g# r# a# d# i# e# n# t# (# 1# 3# 5# d# e# g# ,#  # ## 0# 2# 0# 6# 1# 7#  # 0# %# ,#  # ## 0# 2# 0# 6# 1# 7#  # 1# 0# 0# %# )# ;# 
b# o# r# d# e# r# -# r# a# d# i# u# s# :#  # 1# 4# p# x# ;# 
p# a# d# d# i# n# g# :#  # 1# 8# p# x#  # 2# 4# p# x# ;# 
m# a# r# g# i# n# :#  # 2# 8# p# x#  # 0# ;# 
b# o# x# -# s# h# a# d# o# w# :#  # 0#  # 1# 2# p# x#  # 3# 0# p# x#  # r# g# b# a# (# 0# ,# 0# ,# 0# ,# 0# .# 6# )# ;# 
f# o# n# t# -# f# a# m# i# l# y# :#  # A# r# i# a# l# ,#  # H# e# l# v# e# t# i# c# a# ,#  # s# a# n# s# -# s# e# r# i# f# ;# 
"# ># 

<# h# 2#  # s# t# y# l# e# =# "# 
m# a# r# g# i# n# :#  # 0# ;# 
f# o# n# t# -# s# i# z# e# :#  # 2# 6# p# x# ;# 
f# o# n# t# -# w# e# i# g# h# t# :#  # 7# 0# 0# ;# 
c# o# l# o# r# :#  # ## e# 5# e# 7# e# b# ;# 
l# e# t# t# e# r# -# s# p# a# c# i# n# g# :#  # 0# .# 6# p# x# ;# 
"# ># 
🤖#  # M# o# d# e# l# i# n# g#  # &#  # P# r# e# d# i# c# t# i# o# n# 
<# /# h# 2# ># 

<# /# d# i# v# >


In [23]:
# Time-aware split: train -> validation tail -> test tail
max_date = df_model['Date'].max()
test_start_date = max_date - pd.Timedelta(days=84)
valid_start_date = test_start_date - pd.Timedelta(days=28)

train_mask = df_model['Date'] <= valid_start_date
valid_mask = (df_model['Date'] > valid_start_date) & (df_model['Date'] <= test_start_date)
test_mask = df_model['Date'] > test_start_date

y = np.log1p(df_model['Demand'])
X = df_model.drop(columns=['Demand'])

X_train = X.loc[train_mask].drop(columns=['Date'])
X_valid = X.loc[valid_mask].drop(columns=['Date'])
X_test = X.loc[test_mask].drop(columns=['Date'])

y_train = y.loc[train_mask]
y_valid = y.loc[valid_mask]
y_test = y.loc[test_mask]

cat_cols = [
    c for c in ['Store ID', 'Product ID', 'Category', 'Region', 'Weather Condition', 'Seasonality']
    if c in X_train.columns
]

meta_valid = df_model.loc[valid_mask, ['Date', 'Category', 'Region', 'Seasonality', 'Store ID', 'Product ID']].reset_index(drop=True)
meta_test = df_model.loc[test_mask, ['Date', 'Category', 'Region', 'Seasonality', 'Store ID', 'Product ID']].reset_index(drop=True)

print('Train range:', df_model.loc[train_mask, 'Date'].min().date(), 'to', df_model.loc[train_mask, 'Date'].max().date())
print('Valid range:', df_model.loc[valid_mask, 'Date'].min().date(), 'to', df_model.loc[valid_mask, 'Date'].max().date())
print('Test range :', df_model.loc[test_mask, 'Date'].min().date(), 'to', df_model.loc[test_mask, 'Date'].max().date())
print('Rows -> Train:', X_train.shape[0], ', Valid:', X_valid.shape[0], ', Test:', X_test.shape[0])

# TargetEncoder remains inside each model pipeline so folds do not leak target statistics.

Train range: 2022-01-31 to 2023-10-10
Valid range: 2023-10-11 to 2023-11-07
Test range : 2023-11-08 to 2024-01-30
Rows -> Train: 61800 , Valid: 2800 , Test: 8400


In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from category_encoders import TargetEncoder
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np
import pandas as pd
import gc

def forecasting_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.clip(y_true, 1, None))) * 100
    wape = (np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))) * 100
    return {'MAE': mae, 'R2': r2, 'MAPE_%': mape, 'WAPE_%': wape}

# Target is already log1p transformed (y). We will use TimeSeriesSplit.
tscv = TimeSeriesSplit(n_splits=3, test_size=28 * 100) # 28 days * 100 store-product series

models = {
    'LightGBM': Pipeline([
        ('encoder', TargetEncoder(cols=cat_cols, smoothing=0.3)),
        ('regressor', LGBMRegressor(random_state=42, n_estimators=500, learning_rate=0.05, verbose=-1))
    ]),
    'XGBoost': Pipeline([
        ('encoder', TargetEncoder(cols=cat_cols, smoothing=0.3)),
        ('regressor', XGBRegressor(random_state=42, n_estimators=500, learning_rate=0.05, objective='reg:squarederror'))
    ]),
    'CatBoost': CatBoostRegressor(random_state=42, iterations=500, learning_rate=0.05, cat_features=cat_cols, verbose=0)
}

results_cv = []

print("Starting Walk-Forward Backtesting (3 folds, 28 days each)...")
# Note: we use the original un-split X and y for backtesting.
for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    X_train_cv, y_train_cv = X.iloc[train_idx].drop(columns=['Date']), y.iloc[train_idx]
    X_test_cv, y_test_cv = X.iloc[test_idx].drop(columns=['Date']), y.iloc[test_idx]
    
    y_test_true = np.expm1(y_test_cv)
    
    # Naive Baseline (Lag 7 - Seasonal Naive)
    naive_pred = np.expm1(X_test_cv['lag_7'])
    # Fallback to lag_1 if lag_7 is nan
    naive_pred = naive_pred.fillna(np.expm1(X_test_cv['lag_1'])).fillna(y_test_true.mean())
    
    metrics = forecasting_metrics(y_test_true, naive_pred)
    metrics['Model'] = 'Naive Baseline (Lag 7)'
    metrics['Fold'] = fold + 1
    results_cv.append(metrics)
    
    for name, model in models.items():
        if name == 'CatBoost':
            # CatBoost handles NaNs directly, but we fill them for consistency in X
            # Also convert categorical columns to string for CatBoost
            X_train_cb = X_train_cv.copy()
            X_test_cb = X_test_cv.copy()
            for c in cat_cols:
                X_train_cb[c] = X_train_cb[c].astype(str)
                X_test_cb[c] = X_test_cb[c].astype(str)
            model.fit(X_train_cb.fillna(-999), y_train_cv)
            pred = np.expm1(model.predict(X_test_cb.fillna(-999)))
        else:
            model.fit(X_train_cv, y_train_cv)
            pred = np.expm1(model.predict(X_test_cv))
            
        metrics = forecasting_metrics(y_test_true, pred)
        metrics['Model'] = name
        metrics['Fold'] = fold + 1
        results_cv.append(metrics)
        
cv_df = pd.DataFrame(results_cv)
display(cv_df.groupby('Model')[['MAE', 'WAPE_%', 'MAPE_%']].agg(['mean', 'std']).round(3))

# Train final models on the original train split for subsequent artifacts
lgb_best = models['LightGBM'].fit(X_train, y_train)
xgb_best = models['XGBoost'].fit(X_train, y_train)

X_train_cb_final = X_train.copy()
for c in cat_cols:
    X_train_cb_final[c] = X_train_cb_final[c].astype(str)
cat_best = models['CatBoost'].fit(X_train_cb_final.fillna(-999), y_train)

print("Finished training final models on standard train split.")



In [ ]:
# Train Quantile LightGBM models for Prediction Intervals (80% Confidence Interval)
print("Training Quantile LightGBM models for intervals...")
q_lower = LGBMRegressor(objective='quantile', alpha=0.1, random_state=42, n_estimators=300, verbose=-1)
q_upper = LGBMRegressor(objective='quantile', alpha=0.9, random_state=42, n_estimators=300, verbose=-1)

# Encode categoricals for LightGBM
encoder = TargetEncoder(cols=cat_cols, smoothing=0.3)
X_train_enc = encoder.fit_transform(X_train, y_train)
X_test_enc = encoder.transform(X_test)

q_lower.fit(X_train_enc, y_train)
q_upper.fit(X_train_enc, y_train)

pred_lower = np.expm1(q_lower.predict(X_test_enc))
pred_upper = np.expm1(q_upper.predict(X_test_enc))

print(f"Prediction interval coverage on test set: {np.mean((np.expm1(y_test) >= pred_lower) & (np.expm1(y_test) <= pred_upper)):.1%}")



In [ ]:
import shap

print("Generating SHAP explainability plots...")
# Use the trained LightGBM regressor
explainer = shap.TreeExplainer(lgb_best.named_steps['regressor'])
shap_values = explainer(X_test_enc)

# Global Feature Importance
shap.summary_plot(shap_values, X_test_enc)

# Local Explainability (Waterfall plot for a single prediction)
shap.plots.waterfall(shap_values[0])



In [ ]:
import pmdarima as pm
from statsmodels.tsa.statespace.sarimax import SARIMAX

print("Running SARIMA comparison on 3 representative series...")
# Pick 3 representative series
sample_series = df_model.groupby(['Store ID', 'Product ID'])['Demand'].sum().sort_values(ascending=False).index
high_vol = sample_series[0]
med_vol = sample_series[len(sample_series)//2]
low_vol = sample_series[-1]

sarima_results = []
exog_cols = ['Price', 'Discount', 'Promotion'] # Exogenous features to use

for name, (store, product) in zip(['High Volume', 'Medium Volume', 'Low Volume'], [high_vol, med_vol, low_vol]):
    series_df = df_model[(df_model['Store ID'] == store) & (df_model['Product ID'] == product)].copy()
    series_train = series_df[series_df['Date'] <= test_start_date]
    series_test = series_df[series_df['Date'] > test_start_date]
    
    y_train_s = np.log1p(series_train['Demand'])
    y_test_s = np.log1p(series_test['Demand'])
    
    exog_train = series_train[exog_cols] if all(c in series_train.columns for c in exog_cols) else None
    exog_test = series_test[exog_cols] if all(c in series_test.columns for c in exog_cols) else None
    
    # Simple auto_arima to find parameters quickly
    try:
        model = pm.auto_arima(y_train_s, exogenous=exog_train, seasonal=True, m=7, suppress_warnings=True, max_p=2, max_q=2)
        pred = model.predict(n_periods=len(y_test_s), exogenous=exog_test)
        pred_expm1 = np.expm1(pred)
        
        # Compare with Global LightGBM on the exact same test instances
        X_test_s = series_test.drop(columns=['Demand', 'Date'])
        lgb_pred_expm1 = np.expm1(lgb_best.predict(X_test_s))
        
        y_test_true_s = np.expm1(y_test_s)
        
        sarima_mae = mean_absolute_error(y_test_true_s, pred_expm1)
        lgb_mae = mean_absolute_error(y_test_true_s, lgb_pred_expm1)
        
        sarima_results.append({'Series': name, 'SARIMAX MAE': sarima_mae, 'Global LGBM MAE': lgb_mae})
    except Exception as e:
        print(f"Failed SARIMA on {name}: {e}")

sarima_df = pd.DataFrame(sarima_results)
display(sarima_df)
print("Conclusion: Global Gradient Boosting typically outperforms independent SARIMAX models by sharing cross-product patterns, and is much faster to run on 100+ series.")



## Robustness Audit (Extension)

Rather than a demographic fairness audit (which does not apply to demand forecasting), we perform a **robustness audit across business segments**. 
The WAPE segment diagnostics below ensure our model performs consistently across different Regions, Categories, and Seasons without systemic under-forecasting (which leads to stockouts) or over-forecasting (which leads to waste) in any specific group.



In [25]:
from sklearn.metrics import mean_absolute_error, r2_score

def forecasting_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.clip(y_true, 1, None))) * 100
    wape = (np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))) * 100

    return {
        'MAE': mae,
        'R2': r2,
        'MAPE_%': mape,
        'WAPE_%': wape
    }

y_valid_true = np.expm1(y_valid)
y_test_true = np.expm1(y_test)

valid_lgb_pred = np.expm1(lgb_best.predict(X_valid))
valid_xgb_pred = np.expm1(xgb_best.predict(X_valid))

test_lgb_pred = np.expm1(lgb_best.predict(X_test))
test_xgb_pred = np.expm1(xgb_best.predict(X_test))

results = []

for model_name, pred in [('LightGBM', test_lgb_pred), ('XGBoost', test_xgb_pred)]:
    row = {'Model': model_name}
    row.update(forecasting_metrics(y_test_true, pred))
    results.append(row)

results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
display(results_df)

# Segment diagnostics to detect systematic errors across business slices.
test_diag = meta_test.copy()
test_diag['actual'] = y_test_true.values
test_diag['pred_lgb'] = test_lgb_pred
test_diag['pred_xgb'] = test_xgb_pred

def segment_wape(diag_df, pred_col, by_col):
    grp = diag_df.groupby(by_col).agg(
        abs_err=(pred_col, lambda s: np.abs(s - diag_df.loc[s.index, 'actual']).sum()),
        actual_sum=('actual', 'sum')
    )
    grp['WAPE_%'] = (grp['abs_err'] / grp['actual_sum']) * 100
    return grp[['WAPE_%']].sort_values('WAPE_%', ascending=False)

print('Top segment WAPE by Category (LGBM):')
display(segment_wape(test_diag, 'pred_lgb', 'Category').head())

print('Top segment WAPE by Region (LGBM):')
display(segment_wape(test_diag, 'pred_lgb', 'Region').head())

print('Top segment WAPE by Seasonality (LGBM):')
display(segment_wape(test_diag, 'pred_lgb', 'Seasonality').head())

,Model,MAE,R2,MAPE_%,WAPE_%
0,LightGBM,8.122938,0.925110,12.227574,7.877707
1,XGBoost,10.923509,0.877744,15.053191,10.593728


Top segment WAPE by Category (LGBM):


,WAPE_%
Category,
Toys,9.287474
Groceries,8.509646
Electronics,7.330041
Clothing,6.893849
Furniture,6.265053


Top segment WAPE by Region (LGBM):


,WAPE_%
Region,
West,8.222449
East,7.992445
North,7.926123
South,7.337314


Top segment WAPE by Seasonality (LGBM):


,WAPE_%
Seasonality,
Winter,8.262455
Autumn,6.905584


In [26]:
plot_metrics = results_df.melt(id_vars='Model', value_vars=['MAE', 'WAPE_%', 'MAPE_%'],
                               var_name='Metric', value_name='Value')

plt.figure(figsize=(12, 5))
ax = sns.barplot(data=plot_metrics, x='Metric', y='Value', hue='Model', palette='viridis')
plt.title('Model Error Metrics on Chronological Test Set')
plt.ylabel('Metric Value')
plt.grid(axis='y', alpha=0.2)

for p in ax.patches:
    height = p.get_height()
    if np.isfinite(height):
        ax.annotate(f"{height:.2f}",
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [27]:
from joblib import dump

# Tune blend weight on validation tail instead of fixed 50/50.
weights = np.linspace(0.0, 1.0, 21)
blend_trials = []

for w in weights:
    valid_blend = w * valid_lgb_pred + (1 - w) * valid_xgb_pred
    blend_trials.append((w, mean_absolute_error(y_valid_true, valid_blend)))

best_w, best_valid_mae = min(blend_trials, key=lambda x: x[1])
print(f'Best blend weight for LGBM: {best_w:.2f} (validation MAE={best_valid_mae:.4f})')

ensemble_pred = best_w * test_lgb_pred + (1 - best_w) * test_xgb_pred
ensemble_row = {'Model': 'Blended Ensemble'}
ensemble_row.update(forecasting_metrics(y_test_true, ensemble_pred))

results_final_df = pd.concat([results_df, pd.DataFrame([ensemble_row])], ignore_index=True)
results_final_df = results_final_df.sort_values('MAE').reset_index(drop=True)
display(results_final_df)

# Predicted vs actual trend over time (daily aggregation on the test horizon).
trend_plot = meta_test[['Date']].copy()
trend_plot['actual'] = y_test_true.values
trend_plot['ensemble'] = ensemble_pred
trend_plot['lgbm'] = test_lgb_pred
trend_plot['xgboost'] = test_xgb_pred

daily_trend = trend_plot.groupby('Date', as_index=False)[['actual', 'ensemble', 'lgbm', 'xgboost']].mean()

plt.figure(figsize=(14, 5))
plt.plot(daily_trend['Date'], daily_trend['actual'], label='Actual', linewidth=2, color='black')
plt.plot(daily_trend['Date'], daily_trend['ensemble'], label='Ensemble Forecast', linewidth=2, color='tab:blue')
plt.plot(daily_trend['Date'], daily_trend['lgbm'], label='LGBM', alpha=0.6, linestyle='--')
plt.plot(daily_trend['Date'], daily_trend['xgboost'], label='XGBoost', alpha=0.6, linestyle='--')
plt.title('Daily Actual vs Forecast on Test Horizon')
plt.xlabel('Date')
plt.ylabel('Demand')
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

artifacts = {
    'lgbm_model': lgb_best,
    'xgb_model': xgb_best,
    'blend_weight_lgbm': best_w,
    'feature_columns': X_train.columns.tolist(),
    'categorical_columns': cat_cols
}

dump(artifacts, 'demand_forecasting_artifacts.joblib')
print('Saved model artifacts to demand_forecasting_artifacts.joblib')

Best blend weight for LGBM: 0.90 (validation MAE=8.5977)


,Model,MAE,R2,MAPE_%,WAPE_%
0,Blended Ensemble,8.115920,0.925030,12.222110,7.870901
1,LightGBM,8.122938,0.925110,12.227574,7.877707
2,XGBoost,10.923509,0.877744,15.053191,10.593728


Saved model artifacts to demand_forecasting_artifacts.joblib


## Fast Early-Stopping Alternative (Lower Compute)
This section trains LightGBM and XGBoost with early stopping on the validation tail.
It is a faster alternative to broad RandomizedSearchCV and often reaches similar or better MAE with less compute.

In [30]:
import gc
import numpy as np
import pandas as pd

from sklearn.model_selection import ParameterSampler
from sklearn.metrics import mean_absolute_error
from category_encoders import TargetEncoder
from lightgbm import LGBMRegressor, early_stopping
from xgboost import XGBRegressor
from joblib import dump

# ---------------- Encode categorical variables ---------------- #

fast_encoder = TargetEncoder(cols=cat_cols, smoothing=0.3)

X_train_enc = fast_encoder.fit_transform(X_train.copy(), y_train)
X_valid_enc = fast_encoder.transform(X_valid.copy())
X_test_enc = fast_encoder.transform(X_test.copy())

# ---------------- Hyperparameter Tuning Function ---------------- #

def tune_with_early_stopping(model_type="lgbm", n_iter=10, random_state=42):

    if model_type == "lgbm":

        param_grid = {
            "learning_rate": [0.02, 0.05, 0.08],
            "num_leaves": [31, 50, 70],
            "max_depth": [-1, 5, 10],
            "min_child_samples": [10, 20, 30],
            "subsample": [0.7, 0.85, 1.0],
            "colsample_bytree": [0.7, 0.85, 1.0],
            "reg_alpha": [0.0, 0.1, 0.3],
            "reg_lambda": [0.0, 0.1, 0.3],
        }

        sampler = ParameterSampler(
            param_grid,
            n_iter=n_iter,
            random_state=random_state
        )

        best_model = None
        best_mae = float("inf")
        best_params = None

        for params in sampler:

            model = LGBMRegressor(
                n_estimators=1000,
                random_state=42,
                verbose=-1,
                **params
            )

            model.fit(
                X_train_enc,
                y_train,
                eval_set=[(X_valid_enc, y_valid)],
                eval_metric="l1",
                callbacks=[early_stopping(50, verbose=False)]
            )

            pred = np.expm1(model.predict(X_valid_enc))

            mae = mean_absolute_error(
                np.expm1(y_valid),
                pred
            )

            if mae < best_mae:
                best_mae = mae
                best_model = model
                best_params = params

            gc.collect()

        return best_model, best_mae, best_params

    # ---------------- XGBoost ---------------- #

    param_grid = {
        "learning_rate": [0.02, 0.05, 0.08],
        "max_depth": [4, 6, 8],
        "min_child_weight": [1, 3, 5],
        "subsample": [0.7, 0.85, 1.0],
        "colsample_bytree": [0.7, 0.85, 1.0],
        "reg_alpha": [0.0, 0.1, 0.3],
        "reg_lambda": [0.0, 0.1, 0.3],
    }

    sampler = ParameterSampler(
        param_grid,
        n_iter=n_iter,
        random_state=random_state
    )

    best_model = None
    best_mae = float("inf")
    best_params = None

    for params in sampler:

        model = XGBRegressor(
            objective="reg:squarederror",
            random_state=42,
            n_estimators=300,
            max_bin=256,
            tree_method="hist",
            n_jobs=1,
            verbosity=0,
            **params
        )

        model.fit(
            X_train_enc,
            y_train,
            eval_set=[(X_valid_enc, y_valid)],
            verbose=False
        )

        pred = np.expm1(model.predict(X_valid_enc))

        mae = mean_absolute_error(
            np.expm1(y_valid),
            pred
        )

        if mae < best_mae:
            best_mae = mae
            best_model = model
            best_params = params

        del model
        gc.collect()

    return best_model, best_mae, best_params


# ---------------- Train Models ---------------- #

fast_lgb_model, fast_lgb_val_mae, fast_lgb_params = tune_with_early_stopping(
    model_type="lgbm",
    n_iter=10,
    random_state=42
)

fast_xgb_model, fast_xgb_val_mae, fast_xgb_params = tune_with_early_stopping(
    model_type="xgb",
    n_iter=10,
    random_state=42
)

print("Fast LGBM Validation MAE :", round(fast_lgb_val_mae, 4))
print("Fast XGB Validation MAE  :", round(fast_xgb_val_mae, 4))

# ---------------- Test Predictions ---------------- #

fast_lgb_test = np.expm1(fast_lgb_model.predict(X_test_enc))
fast_xgb_test = np.expm1(fast_xgb_model.predict(X_test_enc))

fast_results = []

for name, pred in [
    ("Fast LGBM", fast_lgb_test),
    ("Fast XGB", fast_xgb_test),
]:
    row = {"Model": name}
    row.update(forecasting_metrics(y_test_true, pred))
    fast_results.append(row)

fast_results_df = (
    pd.DataFrame(fast_results)
    .sort_values("MAE")
    .reset_index(drop=True)
)

display(fast_results_df)

# ---------------- Save Models ---------------- #

fast_artifacts = {
    "encoder": fast_encoder,
    "lgbm_model": fast_lgb_model,
    "xgb_model": fast_xgb_model,
    "lgbm_best_params": fast_lgb_params,
    "xgb_best_params": fast_xgb_params,
    "feature_columns": X_train.columns.tolist(),
    "categorical_columns": cat_cols,
}

dump(fast_artifacts, "demand_forecasting_fast_artifacts.joblib")

print("Saved fast artifacts successfully.")

Fast LGBM Validation MAE : 8.5091
Fast XGB Validation MAE  : 13.2408


,Model,MAE,R2,MAPE_%,WAPE_%
0,Fast LGBM,8.239337,0.922342,12.325936,7.990592
1,Fast XGB,14.129087,0.807258,19.529318,13.702531


Saved fast artifacts successfully.


In [ ]:
from sklearn.model_selection import ParameterSampler
from lightgbm import early_stopping

# Encode categoricals once using train-only statistics, then tune boosted trees with early stopping.
fast_encoder = TargetEncoder(cols=cat_cols, smoothing=0.3)
X_train_enc = fast_encoder.fit_transform(X_train.copy(), y_train)
X_valid_enc = fast_encoder.transform(X_valid.copy())
X_test_enc = fast_encoder.transform(X_test.copy())

def tune_with_early_stopping(model_type='lgbm', n_iter=10, random_state=42):
    if model_type == 'lgbm':
        param_grid = {
            'learning_rate': [0.02, 0.05, 0.08],
            'num_leaves': [31, 50, 70],
            'max_depth': [-1, 5, 10],
            'min_child_samples': [10, 20, 30],
            'subsample': [0.7, 0.85, 1.0],
            'colsample_bytree': [0.7, 0.85, 1.0],
            'reg_alpha': [0.0, 0.1, 0.3],
            'reg_lambda': [0.0, 0.1, 0.3],
        }
        sampler = list(ParameterSampler(param_grid, n_iter=n_iter, random_state=random_state))
        best_model, best_mae, best_params = None, float('inf'), None

        for params in sampler:
            model = LGBMRegressor(
                n_estimators=3000,
                random_state=42,
                verbose=-1,
                **params
            )
            model.fit(
                X_train_enc, y_train,
                eval_set=[(X_valid_enc, y_valid)],
                eval_metric='l1',
                callbacks=[early_stopping(stopping_rounds=75, verbose=False)]
            )
            pred_valid = np.expm1(model.predict(X_valid_enc))
            mae = mean_absolute_error(np.expm1(y_valid), pred_valid)

            if mae < best_mae:
                best_mae = mae
                best_model = model
                best_params = params

        return best_model, best_mae, best_params

    param_grid = {
        'learning_rate': [0.02, 0.05, 0.08],
        'max_depth': [4, 6, 8],
        'min_child_weight': [1, 3, 5],
        'subsample': [0.7, 0.85, 1.0],
        'colsample_bytree': [0.7, 0.85, 1.0],
        'reg_alpha': [0.0, 0.1, 0.3],
        'reg_lambda': [0.0, 0.1, 0.3],
    }
    sampler = list(ParameterSampler(param_grid, n_iter=n_iter, random_state=random_state))
    best_model, best_mae, best_params = None, float('inf'), None

    for params in sampler:
        model = XGBRegressor(
            n_estimators=3000,
            random_state=42,
            objective='reg:squarederror',
            **params
        )
        model.fit(
            X_train_enc, y_train,
            eval_set=[(X_valid_enc, y_valid)],
            verbose=False
        )
        pred_valid = np.expm1(model.predict(X_valid_enc))
        mae = mean_absolute_error(np.expm1(y_valid), pred_valid)

        if mae < best_mae:
            best_mae = mae
            best_model = model
            best_params = params

    return best_model, best_mae, best_params

fast_lgb_model, fast_lgb_val_mae, fast_lgb_params = tune_with_early_stopping('lgbm', n_iter=10, random_state=42)
fast_xgb_model, fast_xgb_val_mae, fast_xgb_params = tune_with_early_stopping('xgb', n_iter=10, random_state=42)

print('Fast LGBM valid MAE:', round(fast_lgb_val_mae, 4))
print('Fast XGB valid MAE :', round(fast_xgb_val_mae, 4))

fast_lgb_test = np.expm1(fast_lgb_model.predict(X_test_enc))
fast_xgb_test = np.expm1(fast_xgb_model.predict(X_test_enc))

fast_results = []
for model_name, pred in [('Fast LGBM', fast_lgb_test), ('Fast XGB', fast_xgb_test)]:
    row = {'Model': model_name}
    row.update(forecasting_metrics(y_test_true, pred))
    fast_results.append(row)

fast_results_df = pd.DataFrame(fast_results).sort_values('MAE').reset_index(drop=True)
display(fast_results_df)

# Save a dedicated fast-inference bundle using encoded-model workflow.
fast_artifacts = {
    'encoder': fast_encoder,
    'lgbm_model': fast_lgb_model,
    'xgb_model': fast_xgb_model,
    'lgbm_best_params': fast_lgb_params,
    'xgb_best_params': fast_xgb_params,
    'feature_columns': X_train.columns.tolist(),
    'categorical_columns': cat_cols
}
dump(fast_artifacts, 'demand_forecasting_fast_artifacts.joblib')
print('Saved fast artifacts to demand_forecasting_fast_artifacts.joblib')

Fast LGBM valid MAE: 7.0098
Fast XGB valid MAE : 9.1515


,Model,MAE,R2,MAPE_%,WAPE_%
0,Fast LGBM,6.571912,0.947862,10.252438,6.373507
1,Fast XGB,9.384467,0.901722,13.371769,9.101151


Saved fast artifacts to demand_forecasting_fast_artifacts.joblib


## Reusable Inference Helper
Use this helper to load saved artifacts and generate demand predictions on new, already-engineered feature rows.

In [32]:
from joblib import load

def _align_inference_features(df_input, feature_columns):
    df_inf = df_input.copy()

    if 'Demand' in df_inf.columns:
        df_inf = df_inf.drop(columns=['Demand'])
    if 'Date' in df_inf.columns:
        df_inf = df_inf.drop(columns=['Date'])

    for col in feature_columns:
        if col not in df_inf.columns:
            df_inf[col] = 0

    extra_cols = [c for c in df_inf.columns if c not in feature_columns]
    if extra_cols:
        df_inf = df_inf.drop(columns=extra_cols)

    return df_inf[feature_columns]

def predict_demand(df_features, artifacts_path='demand_forecasting_artifacts.joblib'):
    artifacts = load(artifacts_path)
    feature_columns = artifacts['feature_columns']

    X_inf = _align_inference_features(df_features, feature_columns)

    # Pipeline artifacts: models already include target encoder step.
    if 'encoder' not in artifacts:
        lgb_pred = np.expm1(artifacts['lgbm_model'].predict(X_inf))
        xgb_pred = np.expm1(artifacts['xgb_model'].predict(X_inf))
        w = artifacts.get('blend_weight_lgbm', 0.5)
        ensemble = w * lgb_pred + (1 - w) * xgb_pred
    else:
        # Fast artifacts: encode first, then run raw models.
        X_inf_enc = artifacts['encoder'].transform(X_inf)
        lgb_pred = np.expm1(artifacts['lgbm_model'].predict(X_inf_enc))
        xgb_pred = np.expm1(artifacts['xgb_model'].predict(X_inf_enc))
        ensemble = 0.5 * lgb_pred + 0.5 * xgb_pred

    pred_df = pd.DataFrame({
        'prediction_lgbm': lgb_pred,
        'prediction_xgboost': xgb_pred,
        'prediction_ensemble': ensemble
    })
    return pred_df

# Example usage (uncomment):
# new_preds = predict_demand(df_model.tail(20))
# display(new_preds.head())

In [ ]:
def predict_recursive(df_history, future_dates, model_pipeline, feature_cols):
    '''
    Recursive forecasting: Predicts day t, appends prediction as actual, and computes lag_1/rolling stats for day t+1.
    df_history: The recent historical dataframe (needs at least 30 days to compute rolling_30/lag_30).
    future_dates: List of future dates to predict.
    '''
    df_curr = df_history.copy()
    
    # Ensure it's sorted by Date, Store, Product
    df_curr = df_curr.sort_values(by=['Date', 'Store ID', 'Product ID']).reset_index(drop=True)
    
    predictions = []
    
    for current_date in future_dates:
        # Create a new row for each Store/Product for the current_date
        stores_products = df_curr[['Store ID', 'Product ID']].drop_duplicates()
        new_rows = stores_products.copy()
        new_rows['Date'] = current_date
        
        # For simplicity, we carry forward static features (Category, Region) and exogenous features (Price, Discount) from the last known date.
        # In production, exogenous features (Price, Promotions) for future dates would be provided by business inputs!
        static_features = df_curr.groupby(['Store ID', 'Product ID']).last().reset_index()
        for col in ['Category', 'Region', 'Price', 'Competitor Pricing', 'Discount', 'Promotion', 'Epidemic', 'Weather Condition', 'Seasonality']:
            if col in static_features.columns:
                new_rows[col] = static_features[col]
                
        # Append to df_curr to compute lags and rolling stats
        df_curr = pd.concat([df_curr, new_rows], ignore_index=True)
        
        # Compute lags and rolling features for the newly added rows
        # Lag 1
        df_curr['lag_1'] = df_curr.groupby(['Store ID', 'Product ID'])['Demand'].shift(1)
        df_curr['lag_7'] = df_curr.groupby(['Store ID', 'Product ID'])['Demand'].shift(7)
        df_curr['lag_30'] = df_curr.groupby(['Store ID', 'Product ID'])['Demand'].shift(30)
        
        df_curr['rolling_mean_7'] = df_curr.groupby(['Store ID', 'Product ID'])['Demand'].transform(lambda x: x.shift(1).rolling(7).mean())
        df_curr['rolling_mean_30'] = df_curr.groupby(['Store ID', 'Product ID'])['Demand'].transform(lambda x: x.shift(1).rolling(30).mean())
        
        # Filter for only the rows of current_date to predict
        current_X = df_curr[df_curr['Date'] == current_date].copy()
        
        # Ensure all feature cols are present (filling missing with 0 or mean)
        for col in feature_cols:
            if col not in current_X.columns:
                current_X[col] = 0
                
        pred_X = current_X[feature_cols]
        
        # Predict using the pipeline
        preds = np.expm1(model_pipeline.predict(pred_X))
        
        # Update the 'Demand' in df_curr with the predictions
        df_curr.loc[df_curr['Date'] == current_date, 'Demand'] = preds
        
        # Store predictions
        current_X['Forecast'] = preds
        predictions.append(current_X[['Date', 'Store ID', 'Product ID', 'Forecast']])
        
    return pd.concat(predictions, ignore_index=True)

print("Recursive forecasting function defined. Ready for backend implementation.")
# Example usage:
# future_dates = pd.date_range(start=df_model['Date'].max() + pd.Timedelta(days=1), periods=7)
# preds = predict_recursive(df_model.tail(100*40), future_dates, lgb_best, X_train.columns)
# display(preds.head())

